In [1]:
!pip install --q -U \
    langchain-openai \
    langchain-chroma \
    langchain-huggingface \
    langchain-core \
    langchain-community \
    chromadb \
    sentence-transformers \
    python-dotenv \
    gradio

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT_PATH = "/content/drive/MyDrive/AI Engineering"

os.chdir(PROJECT_PATH)
sys.path.insert(0, PROJECT_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [41]:
from implementation.answer import fetch_context

docs = fetch_context("Who is Chaitanya?")

print("Retrieved documents:", len(docs))

for i, doc in enumerate(docs[:3], 1):
    print(f"\nDocument {i}:")
    print(doc.page_content[:300])
    print("Metadata:", doc.metadata)

Retrieved documents: 10

Document 1:
# HR Record

# Priya Sharma

## Summary
- **Date of Birth:** January 8, 1986
- **Job Title:** Senior Data Scientist
- **Location:** San Francisco, California
- **Current Salary:** $145,000
Metadata: {'doc_type': 'employees', 'source': '/content/drive/MyDrive/AI Engineering/knowledge-base/employees/Priya Sharma.md'}

Document 2:
---

**Signatures:**

_________________________________
**Michael Torres**
**Title**: Chief Revenue Officer
**Insurellm, Inc.**
**Date**: February 1, 2025

_________________________________
**Robert Chen**
**Title**: Chief Underwriting Officer
**Fortress Business Underwriters**
**Date**: February 1,
Metadata: {'doc_type': 'contracts', 'source': '/content/drive/MyDrive/AI Engineering/knowledge-base/contracts/Contract with Fortress Business Underwriters for Bizllm.md'}

Document 3:
---

**Signatures:**

_________________________________
**Michael Torres**
**Title**: Chief Revenue Officer
**Insurellm, Inc.**
**Date**: January 15

In [50]:
import importlib
import implementation.nemotron_embeddings as nemotron_module
import implementation.ingest as ingest_module

importlib.reload(nemotron_module)
importlib.reload(ingest_module)

print("Nemotron embedding module reloaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Nemotron embedding module reloaded.


In [51]:
import importlib
import implementation.ingest as ingest_module
import implementation.answer as answer_module

importlib.reload(ingest_module)
importlib.reload(answer_module)

print("Modules reloaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modules reloaded!


In [32]:
documents = ingest_module.fetch_documents()
chunks = ingest_module.create_chunks(documents)

vectorstore = ingest_module.create_embeddings(chunks)

print("Final vector count:", vectorstore._collection.count())

Loaded 76 documents.
Created 970 chunks.
There are 970 vectors with 384 dimensions in the vector store.
Final vector count: 970


In [35]:
import importlib
import implementation.answer as answer_module

# Reload answer.py and reconnect to the current ChromaDB collection
answer_module = importlib.reload(answer_module)

print("Vector database:", answer_module.DB_NAME)
print("Retriever reconnected successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector database: /content/drive/MyDrive/AI Engineering/vector_db
Retriever reconnected successfully!


In [36]:
collection = answer_module.vectorstore._collection

print("Collection name:", collection.name)
print("Stored vectors:", collection.count())

Collection name: langchain
Stored vectors: 970


In [37]:
docs = answer_module.fetch_context(
    "What insurance services does Insurellm provide?"
)

print("Retrieved documents:", len(docs))

for i, doc in enumerate(docs[:3], 1):
    print(f"\nDocument {i}:")
    print(doc.page_content[:300])
    print("Metadata:", doc.metadata)

Retrieved documents: 10

Document 1:
# Overview of Insurellm

Insurellm is an innovative insurance tech firm with 32 employees operating primarily remotely across the US, with offices in San Francisco (HQ), New York, Austin, Chicago, and Denver.

Founded in 2015, the company has evolved from a high-growth startup to a lean, profitable 
Metadata: {'doc_type': 'company', 'source': '/content/drive/MyDrive/AI Engineering/knowledge-base/company/overview.md'}

Document 2:
3. **Regular Updates:** Insurellm will offer ongoing updates and enhancements to the Homellm platform, including new features and security improvements.

4. **Feedback Implementation:** Insurellm will actively solicit feedback from GreenValley Insurance to ensure Homellm continues to meet their evol
Metadata: {'doc_type': 'contracts', 'source': '/content/drive/MyDrive/AI Engineering/knowledge-base/contracts/Contract with GreenValley Insurance for Homellm.md'}

Document 3:
2. **Training Services**: Insurellm will provide ini

In [52]:

import gradio as gr
from dotenv import load_dotenv

from implementation.answer import answer_question

load_dotenv(override=True)


# --------------------------------------------------
# 1. Format Retrieved Context
# --------------------------------------------------

def format_context(context):

    result = "## 📚 Relevant Context\n\n"

    if not context:
        return result + "*No relevant documents were retrieved.*"

    for i, doc in enumerate(context, 1):

        source = doc.metadata.get("source", "Unknown source")

        result += f"### Document {i}\n"
        result += f"**Source:** `{source}`\n\n"
        result += doc.page_content
        result += "\n\n---\n\n"

    return result


# --------------------------------------------------
# 2. Chat Function
# --------------------------------------------------

def chat(history):

    try:

        if not history:
            return history, "*Ask a question to retrieve context.*"

        last_message = history[-1]["content"]
        prior = history[:-1]

        answer, context = answer_question(
            last_message,
            prior
        )

        history = history + [
            {
                "role": "assistant",
                "content": answer
            }
        ]

        return history, format_context(context)

    except Exception as e:

        print("CHAT ERROR:", repr(e))

        history = history + [
            {
                "role": "assistant",
                "content": (
                    "Sorry, something went wrong. "
                    "Please check the Colab output."
                )
            }
        ]

        return history, "*An error occurred during generation.*"


# --------------------------------------------------
# 3. Add User Message
# --------------------------------------------------

def put_message_in_chatbot(message, history):

    history = history or []

    if not message or not message.strip():
        return "", history

    history = history + [
        {
            "role": "user",
            "content": message.strip()
        }
    ]

    return "", history


# --------------------------------------------------
# 4. Main Application
# --------------------------------------------------

def main():

    theme = gr.themes.Soft(
        font=["Inter", "system-ui", "sans-serif"]
    )

    with gr.Blocks(
        title="Insurellm Expert Assistant"
    ) as ui:

        gr.Markdown(
            """
            # 🏢 Insurellm Expert Assistant

            Your AI-powered assistant for Insurellm.

            Ask questions about products, contracts,
            customers, company information and more.
            """
        )

        with gr.Row():

            # LEFT COLUMN: CHAT
            with gr.Column(scale=1):

                chatbot = gr.Chatbot(
                    label="💬 Conversation",
                    height=600
                )

                # Message textbox and Send button
                with gr.Row():

                    message = gr.Textbox(
                        placeholder="Ask anything about Insurellm...",
                        show_label=False,
                        lines=2,
                        scale=5
                    )

                    send_btn = gr.Button(
                        "Send 🚀",
                        variant="primary",
                        scale=1
                    )


            # RIGHT COLUMN: CONTEXT
            with gr.Column(scale=1):

                context_markdown = gr.Markdown(
                    label="📚 Retrieved Context",
                    value="*Retrieved context will appear here.*",
                    container=True
                )


        # --------------------------------------------------
        # 5. Event Handling
        # --------------------------------------------------

        def submit_message(message, history):

            return put_message_in_chatbot(message, history)

        # Enter key submission
        message.submit(
            submit_message,
            inputs=[message, chatbot],
            outputs=[message, chatbot]
        ).then(
            chat,
            inputs=[chatbot],
            outputs=[chatbot, context_markdown]
        )

        # Send button submission
        send_btn.click(
            submit_message,
            inputs=[message, chatbot],
            outputs=[message, chatbot]
        ).then(
            chat,
            inputs=[chatbot],
            outputs=[chatbot, context_markdown]
        )


    # --------------------------------------------------
    # 6. Launch
    # --------------------------------------------------

    ui.launch(
        share=True,
        debug=True,
        inbrowser=True,
        theme=theme
    )


# --------------------------------------------------
# 7. Run
# --------------------------------------------------

if __name__ == "__main__":
    main()


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://202ef372634e55e3d6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7862 <> https://202ef372634e55e3d6.gradio.live


In [ ]:
Who is Chaitanya?